# Mining High Voltage PyCaret Dashboard

Run the code cell below to launch the interactive PyCaret `evaluate_model` dashboard for `per_voltage = mininghighvoltage`.

This notebook is intended for Jupyter/Notebook UI where widgets render correctly.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import pandas as pd
from pycaret.classification import setup, compare_models, evaluate_model, plot_model

BASE_DIR = Path.cwd()
if not (BASE_DIR / "df_snr_elem.csv").exists():
    BASE_DIR = Path("c:/Users/phuynh/Projects/robotray-main/models_phan")

DATA_PATH = BASE_DIR / "df_snr_elem.csv"
TARGET_COL = "target"
GROUP_COL = "voltage"
GROUP_VALUE = "mininghighvoltage"
SESSION_ID = 123
MODEL_SHORTLIST = ["lightgbm", "rf", "et", "lr", "ridge", "knn"]
RUN_INTERACTIVE_DASHBOARD = False  # Set True only if your widget rendering works.

plots_dir = BASE_DIR / "pycaret_outputs_v2" / "mininghigh_dashboard_plots"
plots_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df_group = df[df[GROUP_COL].astype(str) == GROUP_VALUE].copy()
ignore_cols = [c for c in ["test", "session", GROUP_COL] if c in df_group.columns]

print(f"Running analysis for {GROUP_COL}={GROUP_VALUE}")
print(f"Rows: {len(df_group):,} | Columns: {df_group.shape[1]}")
print(df_group[TARGET_COL].value_counts())

setup(
    data=df_group,
    target=TARGET_COL,
    ignore_features=ignore_cols,
    session_id=SESSION_ID,
    fold=5,
    use_gpu=False,
    verbose=False,
)

best = compare_models(include=MODEL_SHORTLIST, turbo=True)

if RUN_INTERACTIVE_DASHBOARD:
    # In some environments this crashes during notebook rendering.
    evaluate_model(best)
else:
    print("Interactive dashboard disabled; exporting static plots instead...")
    # Change working dir so PyCaret saves plot files into our chosen folder.
    import os
    prev_cwd = os.getcwd()
    os.chdir(plots_dir)
    try:
        for p in ["confusion_matrix", "class_report", "auc", "pr", "feature"]:
            try:
                plot_model(best, plot=p, save=True)
                print(f"Saved plot: {p}")
            except Exception as sub_exc:
                print(f"Could not save plot {p}: {sub_exc}")
    finally:
        os.chdir(prev_cwd)

print(f"Output folder: {plots_dir}")
